In [1]:
#Data Manipulation
import pandas as pd

#Data Processing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline

#ML Stuff
import pennylane as qml
import torch
import torch.nn as nn

#Visualization
import matplotlib.pyplot as plt

import numpy as np

In [2]:

# load dataset
df = pd.read_csv("diabetes_binary_classification.csv")

#drop rows that have bad data or were missing and replaced with a zero
df.drop(df[df['Diastolic blood pressure (mm Hg)'] == 0].index,inplace=True) #remove the dead guys :p
df.drop(df[df['Triceps skin fold thickness (mm)'] == 0].index,inplace=True) #its literally impossible for this to happen
#reset index after removal
df.reset_index(drop=True,inplace=True)


In [10]:
df.head()

,Number of times pregnant,Plasma glucose concentration a 2 hours in an oral glucose tolerance test,Diastolic blood pressure (mm Hg),Triceps skin fold thickness (mm),2-Hour serum insulin (mu U/ml),Body mass index (weight in kg/(height in m)^2),Diabetes pedigree function,Age (years),Class variable (0 or 1)
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,1,89,66,23,94,28.1,0.167,21,0
3,0,137,40,35,168,43.1,2.288,33,1
4,3,78,50,32,88,31.0,0.248,26,1


In [3]:
X = df.drop("Class variable (0 or 1)", axis=1)
y = df["Class variable (0 or 1)"]

In [4]:
# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Reduce to match qubits
n_qubits = 4
pca = PCA(n_components=n_qubits)
X_reduced = pca.fit_transform(X_scaled)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_reduced, y, test_size=0.2, random_state=42
)

# Convert to torch
X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.float32)
y_test_t = torch.tensor(y_test.values, dtype=torch.float32)

In [5]:
n_qubits = 4
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch")
def qnode(inputs, weights):
    # Encode classical data into rotations
    qml.AngleEmbedding(inputs, wires=range(n_qubits))
    
    # Trainable quantum circuit
    qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
    
    # Return single expectation value
    return qml.expval(qml.PauliZ(0))

In [6]:
weight_shapes = {"weights": (3, n_qubits, 3)}
qlayer = qml.qnn.TorchLayer(qnode, weight_shapes)

In [7]:
model = torch.nn.Sequential(
    torch.nn.Linear(X_train.shape[1], 16),
    torch.nn.ReLU(),
    
    torch.nn.Linear(16, n_qubits),  # reduce to match qubits
    
    qlayer,                        # single quantum layer
    
    torch.nn.Sigmoid()             # binary output
)

In [8]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = torch.nn.BCELoss()

for epoch in range(50):
    optimizer.zero_grad()
    
    outputs = model(X_train_t).squeeze()
    loss = loss_fn(outputs, y_train_t)
    
    loss.backward()
    optimizer.step()
    
    print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

Epoch 0, Loss: 0.7098
Epoch 1, Loss: 0.6861
Epoch 2, Loss: 0.6645
Epoch 3, Loss: 0.6450
Epoch 4, Loss: 0.6282
Epoch 5, Loss: 0.6140
Epoch 6, Loss: 0.6025
Epoch 7, Loss: 0.5935
Epoch 8, Loss: 0.5865
Epoch 9, Loss: 0.5812
Epoch 10, Loss: 0.5769
Epoch 11, Loss: 0.5731
Epoch 12, Loss: 0.5696
Epoch 13, Loss: 0.5662
Epoch 14, Loss: 0.5627
Epoch 15, Loss: 0.5593
Epoch 16, Loss: 0.5561
Epoch 17, Loss: 0.5530
Epoch 18, Loss: 0.5502
Epoch 19, Loss: 0.5475
Epoch 20, Loss: 0.5450
Epoch 21, Loss: 0.5426
Epoch 22, Loss: 0.5405
Epoch 23, Loss: 0.5385
Epoch 24, Loss: 0.5369
Epoch 25, Loss: 0.5356
Epoch 26, Loss: 0.5346
Epoch 27, Loss: 0.5338
Epoch 28, Loss: 0.5331
Epoch 29, Loss: 0.5324
Epoch 30, Loss: 0.5317
Epoch 31, Loss: 0.5309
Epoch 32, Loss: 0.5301
Epoch 33, Loss: 0.5292
Epoch 34, Loss: 0.5283
Epoch 35, Loss: 0.5275
Epoch 36, Loss: 0.5267
Epoch 37, Loss: 0.5260
Epoch 38, Loss: 0.5253
Epoch 39, Loss: 0.5247
Epoch 40, Loss: 0.5241
Epoch 41, Loss: 0.5236
Epoch 42, Loss: 0.5231
Epoch 43, Loss: 0.522

In [9]:
from sklearn.metrics import accuracy_score, roc_auc_score

model.eval()
with torch.no_grad():
    probs = model(X_test_t).squeeze()
    preds = (probs >= 0.5).float()

accuracy = accuracy_score(y_test_t.numpy(), preds.numpy())
auc = roc_auc_score(y_test_t.numpy(), probs.numpy())

print(f"Accuracy: {accuracy:.4f}")
print(f"ROC-AUC: {auc:.4f}")

Accuracy: 0.7500
ROC-AUC: 0.7886
